# Step 1 V2 - training-only predictive structure and timing audit

This notebook implements Exercise 1 of *Implied Vol Event Time Series* while treating every arrow as a **Granger-predictive direction**, not a structurally identified cause. It preserves the 17:00 New York model day, session-aware returns, strict complete-session rule, DST/holiday handling, and calendar-gap protection.

The ATM source contains a date but no intraday quote timestamp. Therefore timing is not identified from the file. Two Equation 2 variants are carried through training-only inner validation: `advance` (date-D quote predicts session D+1) and `same_label` (date-D quote is assumed observable before session D). Equation 3 is separate: its target is always the unshifted date-D quote, with every regressor drawn from earlier model days. The single-tenor file cannot implement the conditional forward variance in PDF Equation (5); lagged 1D variance is a material approximation.

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from IPython.display import display, Markdown

SEED=16017; ALPHA=.05; MAX_CAL_GAP=7; HOURS_PER_SESSION=24
TIMINGS=["advance","same_label"]; LAG_CANDIDATES=[1,5]; WEEKDAY_CANDIDATES=[False,True]
BASE=Path(os.environ.get("PIPELINE_BASE", Path.cwd())).resolve()
ATM=Path(os.environ.get("ATM_CSV_OVERRIDE", Path.home()/"Downloads/usdjpy_1d_atm_vol.csv"))
RETURNS=BASE/"Data/processed/usdjpy_hourly_modelday_returns.csv"
RAW=BASE/"Data/usdjpy_hourly_data/usdjpy_hourly_mid.csv"
OUT=BASE/"step1_results_v2"; OUT.mkdir(parents=True,exist_ok=True)
for p in (RAW,RETURNS,ATM): assert p.exists(), f"missing {p}"
rng=np.random.default_rng(SEED)
print("base",BASE); print("seed",SEED)

base /Users/aksharaa/Documents/Codex/2026-08-14/review-and-methodologically-correct-the-following/work/source_mirror
seed 16017


## Panel reconstruction and quote-source audit

The split is defined from the model-day calendar only, before alignment, lag, weekday, or structure selection. No post-split outcome is read by any selection routine.

In [2]:
h=pd.read_csv(RETURNS,parse_dates=["timestamp"],low_memory=False).sort_values("timestamp").reset_index(drop=True)
assert h.timestamp.is_unique
h["model_day"]=pd.to_datetime(h.model_day,errors="coerce")
h["valid"]=h.valid_within_session_return.astype(str).str.lower().eq("true")
ny=h.timestamp.dt.tz_convert("America/New_York")
recomputed=(ny.dt.tz_localize(None)-pd.Timedelta(hours=17)).dt.normalize()+pd.Timedelta(days=1)
has=h.model_day.notna(); assert (recomputed[has]==h.model_day[has]).all()
h["ret"]=np.where(h.valid,h.hourly_log_return,np.nan); h["ret2"]=h.ret**2; h["ret4"]=h.ret**4
g=h[has].groupby("model_day")
panel=pd.DataFrame({"n_bars":g.size(),"n_valid_ret":g.valid.sum().astype(int),"r_raw":g.ret.sum(min_count=1),
                    "rv_raw":g.ret2.sum(min_count=1),"rq_raw":(HOURS_PER_SESSION/3)*g.ret4.sum(min_count=1)}).reset_index()
panel["complete24"]=(panel.n_bars==24)&(panel.n_valid_ret==24)
panel["zero_rv"]=panel.complete24&(panel.rv_raw==0)
panel["rv_eligible"]=panel.complete24&~panel.zero_rv
for c,raw in [("r","r_raw"),("rv","rv_raw"),("rq","rq_raw")]: panel[c]=np.where(panel.rv_eligible,panel[raw],np.nan)
panel["r2"]=panel.r**2
atm=pd.read_csv(ATM); atm["date"]=pd.to_datetime(atm.date); atm["iv_quote"]=(atm.atm_vol_pct/100)**2
assert atm.date.is_unique and atm.iv_quote.gt(0).all()
time_cols=[c for c in atm if any(k in c.lower() for k in ("time","stamp","datetime"))]
assert time_cols==[], f"unexpected timestamp columns: {time_cols}"
panel=panel.merge(atm[["date","iv_quote","cal_days_to_next"]].rename(columns={"date":"model_day"}),how="left",on="model_day")
panel["gap_days"]=panel.model_day.diff().dt.days
panel["contig_break"]=panel.gap_days.isna()|(panel.gap_days>MAX_CAL_GAP)
panel["iv_eq2_advance"]=panel.iv_quote.shift(1); panel.loc[panel.contig_break,"iv_eq2_advance"]=np.nan
panel["iv_eq2_same_label"]=panel.iv_quote
panel["iv_eq3_target"]=panel.iv_quote
cut=(panel.model_day.dt.tz_localize("America/New_York")+pd.Timedelta(hours=17)).dt.tz_convert("UTC")
panel["session_end_utc"]=cut; panel["session_start_utc"]=(panel.model_day-pd.Timedelta(days=1)).dt.tz_localize("America/New_York").add(pd.Timedelta(hours=17)).dt.tz_convert("UTC")
eligible_days=panel.loc[panel.rv_eligible&panel.iv_quote.notna(),"model_day"].sort_values()
SPLIT_DATE=eligible_days.iloc[int(np.floor(.80*len(eligible_days)))]
INNER_DATE=eligible_days[eligible_days<SPLIT_DATE].iloc[int(np.floor(.80*(eligible_days<SPLIT_DATE).sum()))]
panel["sample_role"]=np.where(panel.model_day<SPLIT_DATE,"training","final_test")
assert panel.loc[panel.sample_role.eq("training"),"model_day"].max()<SPLIT_DATE
audit=pd.DataFrame({"quantity":["hourly rows","model days","complete 24h","zero-RV excluded","RV eligible","IV rows","final split","inner split","IV intraday timestamp columns"],
                    "value":[len(h),len(panel),int(panel.complete24.sum()),int(panel.zero_rv.sum()),int(panel.rv_eligible.sum()),int(panel.iv_quote.notna().sum()),str(SPLIT_DATE.date()),str(INNER_DATE.date()),str(time_cols)]})
display(audit); audit.to_csv(OUT/"step1_panel_audit.csv",index=False)
print("ATM timing limitation: date-only source; no intraday quote timestamp. Robustness variants are mandatory.")

,quantity,value
0,hourly rows,149842
1,model days,5842
2,complete 24h,5764
3,zero-RV excluded,2
4,RV eligible,5762
5,IV rows,5517
6,final split,2022-04-12
7,inner split,2018-11-27
8,IV intraday timestamp columns,[]


ATM timing limitation: date-only source; no intraday quote timestamp. Robustness variants are mandatory.


## Explicit economic-time table

`df_offset=0` means the same dataframe row as the target; negative offsets are earlier rows. Under `same_label`, Equation 2's IV regressor is only admissible if the date-D quote was observed before the session-D outcome. Because the source cannot verify that, it remains a robustness variant. Equation 3 uses only offsets -1 or earlier under every specification.

In [3]:
timing_rows=[]
for timing_spec in TIMINGS:
    timing_rows += [
      {"equation":"Eq2","role":"target","variable":"rv_D","economic_time":"session (D-1 17:00 NY, D 17:00 NY]","df_offset":0,"known_before_target":False,"timing_variant":timing_spec},
      {"equation":"Eq2","role":"regressor","variable":"rv/r2 lag j","economic_time":"session ending model day D-j","df_offset":"-j","known_before_target":True,"timing_variant":timing_spec},
      {"equation":"Eq2","role":"regressor","variable":"IV block","economic_time":"quote date D-j" if timing_spec=="advance" else "quote date D-(j-1)","df_offset":"-j" if timing_spec=="advance" else "-(j-1)","known_before_target":True if timing_spec=="advance" else "unverified: date-only quote","timing_variant":timing_spec},
      {"equation":"Eq3","role":"target","variable":"iv_quote_D","economic_time":"unknown intraday time on quote date D","df_offset":0,"known_before_target":False,"timing_variant":timing_spec},
      {"equation":"Eq3","role":"regressor","variable":"r2/rv/iv_quote lag j","economic_time":"model day / quote date D-j","df_offset":"-j","known_before_target":True,"timing_variant":timing_spec},
    ]
timing_table=pd.DataFrame(timing_rows).drop_duplicates(); display(timing_table); timing_table.to_csv(OUT/"step1_timing_table.csv",index=False)
assert timing_table.query("equation=='Eq3' and role=='regressor'").known_before_target.eq(True).all()

,equation,role,variable,economic_time,df_offset,known_before_target,timing_variant
0,Eq2,target,rv_D,"session (D-1 17:00 NY, D 17:00 NY]",0,False,advance
1,Eq2,regressor,rv/r2 lag j,session ending model day D-j,-j,True,advance
2,Eq2,regressor,IV block,quote date D-j,-j,True,advance
3,Eq3,target,iv_quote_D,unknown intraday time on quote date D,0,False,advance
4,Eq3,regressor,r2/rv/iv_quote lag j,model day / quote date D-j,-j,True,advance
5,Eq2,target,rv_D,"session (D-1 17:00 NY, D 17:00 NY]",0,False,same_label
6,Eq2,regressor,rv/r2 lag j,session ending model day D-j,-j,True,same_label
7,Eq2,regressor,IV block,quote date D-(j-1),-(j-1),unverified: date-only quote,same_label
8,Eq3,target,iv_quote_D,unknown intraday time on quote date D,0,False,same_label
9,Eq3,regressor,r2/rv/iv_quote lag j,model day / quote date D-j,-j,True,same_label


## Training-only inner selection

Candidate lag order 5 means lags 1-5 jointly. A positive log-variance model is scored by QLIKE on the inner validation tail. The final test is not accessed.

In [4]:
def offsets(src,tgt,timing,p):
    if src=="iv" and tgt in ("rv","r2") and timing=="same_label": return list(range(0,p))
    return list(range(1,p+1))
def shifted(x,k): return x.shift(k) if k else x
def block(src,tgt,timing,p):
    s=panel.iv_quote if src=="iv" else panel[src]
    return np.column_stack([shifted(s,k) for k in offsets(src,tgt,timing,p)])
def win_ok(maxoff):
    ok=np.ones(len(panel),bool); br=panel.contig_break.to_numpy(bool)
    for k in range(maxoff): ok &= ~np.r_[np.ones(k,bool),br[:len(br)-k]] if k else ~br
    ok[:maxoff]=False; return ok
def design_xy(target,timing,p,weekday,rows=None,blocks=("r2","rv","iv"),log=True):
    y=panel.iv_quote.to_numpy(float) if target=="iv" else panel[target].to_numpy(float)
    X=np.column_stack([block(s,target,timing,p) for s in blocks])
    if weekday: X=np.column_stack([X,pd.get_dummies(panel.model_day.dt.dayofweek,drop_first=True,dtype=float).to_numpy()])
    maxoff=max(max(offsets(s,target,timing,p)) for s in blocks)
    ok=win_ok(maxoff)&np.isfinite(y)&np.isfinite(X).all(1)&(y>0)
    if rows is not None: ok &= rows
    return (np.log(y[ok]) if log else y[ok]),X[ok],np.flatnonzero(ok)
def qlike(y,f):
    z=y/f; return z-np.log(z)-1
sel=[]
for timing in TIMINGS:
  for p in LAG_CANDIDATES:
    for wd in WEEKDAY_CANDIDATES:
      losses=[]
      for target in ("rv","iv"):
        y,X,idx=design_xy(target,timing,p,wd,rows=panel.model_day.lt(SPLIT_DATE).to_numpy())
        tr=panel.model_day.iloc[idx].lt(INNER_DATE).to_numpy(); va=~tr
        m=sm.OLS(y[tr],sm.add_constant(X[tr],has_constant="add")).fit()
        f=np.exp(m.predict(sm.add_constant(X[va],has_constant="add")))
        actual=np.exp(y[va]); losses.append(float(np.mean(qlike(actual,f))))
      sel.append({"timing":timing,"lag":p,"weekday":wd,"eq2_qlike":losses[0],"eq3_qlike":losses[1],"mean_qlike":np.mean(losses)})
selection=pd.DataFrame(sel).sort_values("mean_qlike").reset_index(drop=True)
# A date-only same-label quote may overlap the realised session. Information-set integrity
# therefore fixes the advance convention as primary; same-label remains a robustness variant.
CHOSEN_TIMING="advance"
chosen=selection[selection.timing.eq(CHOSEN_TIMING)].iloc[0]
CHOSEN_LAG=int(chosen.lag); CHOSEN_WEEKDAY=bool(chosen.weekday)
display(selection); selection.to_csv(OUT/"step1_inner_selection.csv",index=False)
assert panel.loc[panel.model_day>=SPLIT_DATE,["rv","iv_quote"]].notna().sum().sum()>0  # existence only; values never passed to selection
print("FROZEN",CHOSEN_TIMING,CHOSEN_LAG,CHOSEN_WEEKDAY)

,timing,lag,weekday,eq2_qlike,eq3_qlike,mean_qlike
0,same_label,5,True,0.360472,0.356727,0.358599
1,same_label,5,False,0.358836,0.379193,0.369015
2,advance,5,True,0.383565,0.356727,0.370146
3,advance,5,False,0.377892,0.379193,0.378543
4,same_label,1,True,0.390550,0.401066,0.395808
5,advance,1,True,0.410809,0.401066,0.405937
6,same_label,1,False,0.387566,0.442802,0.415184
7,advance,1,False,0.405747,0.442802,0.424275


FROZEN advance 5 True


## HAC block-Wald tests and multiplicity

For each source-target direction, the unrestricted regression contains the target's lags, the source block, the remaining state's block, and training-estimated weekday dummies. The null jointly sets every source lag coefficient to zero. HAC covariance handles heteroskedasticity and autocorrelation. Holm adjustment is applied across all 24 timing-by-lag-by-direction tests, not within convenient subfamilies.

In [5]:
STATES=["r2","rv","iv"]
def wald_direction(src,tgt,timing,p):
    ctl=next(s for s in STATES if s not in (src,tgt))
    y=panel.iv_quote.to_numpy(float) if tgt=="iv" else panel[tgt].to_numpy(float)
    own=block(tgt,tgt,timing,p); sb=block(src,tgt,timing,p); cb=block(ctl,tgt,timing,p)
    wd=pd.get_dummies(panel.model_day.dt.dayofweek,drop_first=True,dtype=float).to_numpy()
    X=np.column_stack([own,sb,cb,wd]); maxoff=max(max(offsets(s,tgt,timing,p)) for s in (tgt,src,ctl))
    ok=win_ok(maxoff)&panel.model_day.lt(SPLIT_DATE).to_numpy()&np.isfinite(y)&np.isfinite(X).all(1)
    Xc=sm.add_constant(X[ok],has_constant="add"); m=sm.OLS(y[ok],Xc).fit(cov_type="HAC",cov_kwds={"maxlags":max(5,p)})
    R=np.zeros((sb.shape[1],Xc.shape[1])); start=1+own.shape[1]
    R[:,start:start+sb.shape[1]]=np.eye(sb.shape[1])
    wt=m.wald_test(R,scalar=True)
    mr=sm.OLS(y[ok],sm.add_constant(np.column_stack([own[ok],cb[ok],wd[ok]]),has_constant="add")).fit()
    partial=(mr.ssr-sm.OLS(y[ok],Xc).fit().ssr)/mr.ssr
    return {"source":src,"target":tgt,"direction":f"{src} -> {tgt}","timing":timing,"lag_order":p,"n":int(ok.sum()),
            "wald_chi2":float(wt.statistic),"df":sb.shape[1],"p_raw":float(wt.pvalue),"partial_R2":float(partial)}
tests=pd.DataFrame([wald_direction(s,t,timing,p) for timing in TIMINGS for p in LAG_CANDIDATES for s in STATES for t in STATES if s!=t])
tests["q_holm_all24"]=multipletests(tests.p_raw,method="holm")[1]
tests["reject_holm_5pct"]=tests.q_holm_all24<ALPHA
tests.to_csv(OUT/"step1_robust_wald_all.csv",index=False)
primary=tests[(tests.timing==CHOSEN_TIMING)&(tests.lag_order==CHOSEN_LAG)].copy()
primary["retained"]=primary.reject_holm_5pct
display(primary.sort_values("partial_R2",ascending=False))

,source,target,direction,timing,lag_order,n,wald_chi2,df,p_raw,partial_R2,q_holm_all24,reject_holm_5pct,retained
10,iv,r2,iv -> r2,advance,5,3738,72.972807,5,2.462894e-14,0.270583,4.679499e-13,True,True
11,iv,rv,iv -> rv,advance,5,3738,79.690739,5,9.739817e-16,0.170148,2.045362e-14,True,True
9,rv,iv,rv -> iv,advance,5,3765,5.801866,5,3.259781e-01,0.029728,1.000000e+00,False,False
8,rv,r2,rv -> r2,advance,5,3738,3.034631,5,6.946466e-01,0.003376,1.000000e+00,False,False
6,r2,rv,r2 -> rv,advance,5,3738,2.766471,5,7.359347e-01,0.002577,1.000000e+00,False,False
7,r2,iv,r2 -> iv,advance,5,3765,0.283832,5,9.979361e-01,0.000353,1.000000e+00,False,False


## Conditional nonlinear dependence (exploratory)

This is not a bivariate TDMI gate. For each direction, time-series folds compare a nonlinear model of the target's own history, the third state, and weekday controls against the same model plus the entire source-lag block. Block/circular permutations of the source block provide an exploratory p-value while preserving serial structure. Because this is a model-based conditional importance procedure rather than an established conditional-TDMI estimator, it is explicitly not required for arrow retention.

In [6]:
def nonlinear_check(src,tgt,B=99):
    p=CHOSEN_LAG; timing=CHOSEN_TIMING; ctl=next(s for s in STATES if s not in (src,tgt))
    y=panel.iv_quote.to_numpy(float) if tgt=="iv" else panel[tgt].to_numpy(float)
    S=block(src,tgt,timing,p); C=np.column_stack([block(tgt,tgt,timing,p),block(ctl,tgt,timing,p),
        pd.get_dummies(panel.model_day.dt.dayofweek,drop_first=True,dtype=float).to_numpy()])
    maxoff=max(max(offsets(s,tgt,timing,p)) for s in (src,tgt,ctl)); ok=win_ok(maxoff)&panel.model_day.lt(SPLIT_DATE).to_numpy()&np.isfinite(y)&np.isfinite(S).all(1)&np.isfinite(C).all(1)
    y=np.log(np.maximum(y[ok],1e-12)); S=S[ok]; C=C[ok]; diffs=[]; null=[]
    for tr,va in TimeSeriesSplit(n_splits=4).split(C):
        base=RandomForestRegressor(n_estimators=120,min_samples_leaf=20,max_features=.8,random_state=SEED,n_jobs=-1).fit(C[tr],y[tr])
        full=RandomForestRegressor(n_estimators=120,min_samples_leaf=20,max_features=.8,random_state=SEED+1,n_jobs=-1).fit(np.c_[C[tr],S[tr]],y[tr])
        b=np.mean((y[va]-base.predict(C[va]))**2); f=np.mean((y[va]-full.predict(np.c_[C[va],S[va]]))**2); diffs.append(b-f)
        for _ in range(B//4):
            k=int(rng.integers(5,max(6,len(va)-5))); Sp=np.roll(S[va],k,axis=0)
            null.append(b-np.mean((y[va]-full.predict(np.c_[C[va],Sp]))**2))
    obs=float(np.mean(diffs)); pv=(1+sum(v>=obs for v in null))/(1+len(null))
    return {"direction":f"{src} -> {tgt}","cv_mse_reduction":obs,"relative_reduction":obs/np.var(y),"p_exploratory":pv}
nonlinear=pd.DataFrame([nonlinear_check(s,t) for s in STATES for t in STATES if s!=t])
nonlinear.to_csv(OUT/"step1_conditional_nonlinear_exploratory.csv",index=False); display(nonlinear)

,direction,cv_mse_reduction,relative_reduction,p_exploratory
0,r2 -> rv,0.000204,0.000205,0.371134
1,r2 -> iv,-0.003247,-0.003580,0.298969
2,rv -> r2,-0.002185,-0.000361,0.144330
3,rv -> iv,-0.001959,-0.002159,0.010309
4,iv -> r2,0.093071,0.015366,0.010309
5,iv -> rv,0.071908,0.072189,0.010309


## Training-subsample stability and frozen hand-off

The final structure follows only the selected HAC-Wald family. Stability windows are economic training eras and never touch the final test. The nonlinear table is supporting/exploratory evidence only.

In [7]:
eras=[("pre_GFC_to_outage",pd.Timestamp("2003-01-01"),pd.Timestamp("2010-01-01")),
      ("post_outage_Abenomics",pd.Timestamp("2010-10-01"),pd.Timestamp("2016-01-01")),
      ("recent_training",pd.Timestamp("2016-01-01"),SPLIT_DATE)]
stab=[]
for name,lo,hi in eras:
  for _,r in primary.iterrows():
    # same robust design, restricted to era, implemented inline by temporarily adding an era flag
    src,tgt=r.source,r.target; ctl=next(s for s in STATES if s not in (src,tgt)); p=CHOSEN_LAG
    y=panel.iv_quote.to_numpy(float) if tgt=="iv" else panel[tgt].to_numpy(float)
    own=block(tgt,tgt,CHOSEN_TIMING,p); sb=block(src,tgt,CHOSEN_TIMING,p); cb=block(ctl,tgt,CHOSEN_TIMING,p)
    wd=pd.get_dummies(panel.model_day.dt.dayofweek,drop_first=True,dtype=float).to_numpy(); X=np.c_[own,sb,cb,wd]
    mo=max(max(offsets(s,tgt,CHOSEN_TIMING,p)) for s in (tgt,src,ctl)); ok=win_ok(mo)&panel.model_day.ge(lo).to_numpy()&panel.model_day.lt(hi).to_numpy()&np.isfinite(y)&np.isfinite(X).all(1)
    if ok.sum()<120: continue
    Xc=sm.add_constant(X[ok],has_constant="add"); m=sm.OLS(y[ok],Xc).fit(cov_type="HAC",cov_kwds={"maxlags":max(5,p)})
    R=np.zeros((sb.shape[1],Xc.shape[1])); st=1+own.shape[1]; R[:,st:st+sb.shape[1]]=np.eye(sb.shape[1]); w=m.wald_test(R,scalar=True)
    stab.append({"era":name,"direction":r.direction,"n":int(ok.sum()),"wald_chi2":float(w.statistic),"p_raw":float(w.pvalue)})
stability=pd.DataFrame(stab); stability["q_BH_within_era"]=stability.groupby("era").p_raw.transform(lambda x:multipletests(x,method="fdr_bh")[1])
stability.to_csv(OUT/"step1_stability.csv",index=False); display(stability)

ret=primary.loc[primary.retained,"direction"].tolist(); eq2=[s for s in STATES if s!="rv" and f"{s} -> rv" in ret]+(["rv"] if True else [])
eq3=[s for s in STATES if s!="iv" and f"{s} -> iv" in ret]+["iv"]
panel.to_csv(OUT/"step1_panel_v2.csv",index=False)
primary.to_csv(OUT/"step1_selected_structure.csv",index=False)
decisions={"seed":SEED,"split_date":str(SPLIT_DATE.date()),"inner_validation_start":str(INNER_DATE.date()),
 "final_test_rule":"model_day >= split_date; untouched until Step 2 final evaluation",
 "iv_source_timestamp":"absent; date only","timing_variants":TIMINGS,"chosen_eq2_timing":CHOSEN_TIMING,
 "eq3_target":"unshifted iv_quote on quote date","eq3_regressor_rule":"all dataframe offsets <= -1",
 "primary_lag_order":CHOSEN_LAG,"weekday_adjustment":CHOSEN_WEEKDAY,"selection_metric":"timing fixed to conservative advance information set; lag and weekday chosen by mean inner-validation QLIKE across Eq2 and Eq3 positive log models",
 "multiplicity":"Holm across 24 direction x lag x timing tests","retained_predictive_directions":ret,
 "eq2_blocks_retained":sorted(set(eq2)),"eq3_blocks_retained":sorted(set(eq3)),
 "conditional_nonlinear_role":"exploratory, not retention gate","equation5_forward_variance":"not implemented: only one tenor/ticker and no term structure"}
(OUT/"frozen_decisions.json").write_text(json.dumps(decisions,indent=2))
assert panel.query("sample_role=='final_test'").model_day.min()>=SPLIT_DATE
assert timing_table.query("equation=='Eq3' and role=='regressor'").df_offset.astype(str).str.contains("-j").all()
print(json.dumps(decisions,indent=2))

,era,direction,n,wald_chi2,p_raw,q_BH_within_era
0,pre_GFC_to_outage,r2 -> rv,1003,4.076803,5.384121e-01,5.384121e-01
1,pre_GFC_to_outage,r2 -> iv,990,4.223367,5.177251e-01,5.384121e-01
2,pre_GFC_to_outage,rv -> r2,1003,4.497203,4.802577e-01,5.384121e-01
3,pre_GFC_to_outage,rv -> iv,990,12.066565,3.388698e-02,6.777396e-02
4,pre_GFC_to_outage,iv -> r2,1003,19.733665,1.401999e-03,4.205996e-03
5,pre_GFC_to_outage,iv -> rv,1003,35.225908,1.356251e-06,8.137507e-06
6,post_outage_Abenomics,r2 -> rv,1184,4.018661,5.467324e-01,5.467324e-01
7,post_outage_Abenomics,r2 -> iv,1210,19.750552,1.391825e-03,4.175475e-03
8,post_outage_Abenomics,rv -> r2,1184,7.075014,2.151209e-01,2.581451e-01
9,post_outage_Abenomics,rv -> iv,1210,8.609286,1.257005e-01,1.885507e-01


{
  "seed": 16017,
  "split_date": "2022-04-12",
  "inner_validation_start": "2018-11-27",
  "final_test_rule": "model_day >= split_date; untouched until Step 2 final evaluation",
  "iv_source_timestamp": "absent; date only",
  "timing_variants": [
    "advance",
    "same_label"
  ],
  "chosen_eq2_timing": "advance",
  "eq3_target": "unshifted iv_quote on quote date",
  "eq3_regressor_rule": "all dataframe offsets <= -1",
  "primary_lag_order": 5,
  "weekday_adjustment": true,
  "selection_metric": "timing fixed to conservative advance information set; lag and weekday chosen by mean inner-validation QLIKE across Eq2 and Eq3 positive log models",
  "multiplicity": "Holm across 24 direction x lag x timing tests",
  "retained_predictive_directions": [
    "iv -> r2",
    "iv -> rv"
  ],
  "eq2_blocks_retained": [
    "iv",
    "rv"
  ],
  "eq3_blocks_retained": [
    "iv"
  ],
  "conditional_nonlinear_role": "exploratory, not retention gate",
  "equation5_forward_variance": "not implemen

## Interpretation

The generated `step1_selected_structure.csv` and `frozen_decisions.json` are the only authoritative conclusions. Statistical rejection means conditional predictive content given the other state, own history, and calendar controls. It does not identify an intervention, exclude latent common drivers, or establish structural causality.